In [ ]:
# Import packages needed for all notebooks

import zipfile
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize, differential_evolution
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import norm
from scipy.interpolate import RBFInterpolator
from smac import HyperparameterOptimizationFacade, Scenario
from ConfigSpace import Configuration, ConfigurationSpace, Float
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import LeaveOneOut
import itertools
from plotly.subplots import make_subplots
import torch
from botorch.models import SingleTaskGP
from gpytorch.mlls import ExactMarginalLogLikelihood
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import LogExpectedImprovement, LogNoisyExpectedImprovement
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling import SobolQMCNormalSampler
from botorch.optim import optimize_acqf, gen_batch_initial_conditions
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize
from botorch.generation import get_best_candidates, gen_candidates_torch

In [2]:
# Extract initial_data numpy files from whatever location they're saved in

zip_path = r'C:\Users\chase\Imperial College ML Cert Program\Capstone Project\Initial_data_points_starter.zip'
extract_path = r'C:\Users\chase\Imperial College ML Cert Program\Capstone Project'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(os.listdir(extract_path))

['Capstone Data Minimal Extra Points Sandbox.ipynb', 'Capstone Data.ipynb', 'Initial Setup.ipynb', 'initial_data', 'Initial_data_points_starter.zip', 'smac3_output', '__MACOSX']


In [3]:
# Convert initial_data numpy files to pandas dataframes

base_path = r"C:\Users\chase\Imperial College ML Cert Program\Capstone Project\initial_data"

dfs = {}

for i in range(1, 9):
    folder_name = f'function_{i}'
    inputs_path = os.path.join(base_path, folder_name, 'initial_inputs.npy')
    outputs_path = os.path.join(base_path, folder_name, 'initial_outputs.npy')
    
    if os.path.exists(inputs_path) and os.path.exists(outputs_path):
        X = np.load(inputs_path)
        y = np.load(outputs_path)
        
        # Handle multidimensional input names dynamically
        input_cols = [f'X_{j+1}' for j in range(X.shape[1] if len(X.shape) > 1 else 1)]
        
        df = pd.DataFrame(X, columns=input_cols)
        df['y'] = y
        
        dfs[folder_name] = df

In [5]:
# Dictionaries storing weekly results

new_points_1 = {'X_1': [0.097534, 0.369062, 0.142575, 0.156346, 0.729121, 0.996765, 0.824397, 0.368868, 0.362322, 0.369953, 0.500000, 0.321962, 0.353364],
                'X_2': [0.761249, 0.334110, 0.512408, 0.206162, 0.343076, 0.331699, 0.772848, 0.334997, 0.360779, 0.424178, 0.500000, 0.360593, 0.351886],
                'y': [3.484853029466849e-154, 2.9030338105184675e-8, -2.2562400102825942e-61, 2.3405837646243422e-82, -1.1566686617918571e-64,
                      -3.0771484084535896e-156, 2.571272313588064e-43, 3.689377114039466e-8, 0.0000055322133538987255, -0.006623244612625458,
                      2.6752879910742468e-9, 1.7895550612731168e-10, 2.2215212367561578e-7]
}

new_points_2 = {'X_1': [0.830672, 0.988613, 0.000000, 0.716910, 0.677493, 0.692699, 0.704602, 0.687711, 0.702597, 0.695612, 0.698378, 0.698767, 0.696082],
                'X_2': [0.990920, 0.000000, 0.999999, 0.000000, 0.999999, 0.173900, 0.185972, 0.173241, 0.170176, 0.418475, 0.173130, 0.175588, 0.171817],
                'y': [0.15472125342388554, 0.015063676917738713, 0.0550726792848499, 0.5354474341600155, 0.5167965072531588, 0.7110438023772423,
                      0.677367764230634, 0.667005916275078, 0.7010036427605973, 0.6050892632976823, 0.7295854282471306, 0.5697162961837035,
                      0.49119398436996925]
}

new_points_3 = {'X_1': [0.425917, 0.142575, 0.003733, 0.996765, 0.481488, 0.969757, 0.596284, 0.454749, 0.385388, 0.413471, 0.454429, 0.469700, 0.527595],
                'X_2': [0.387187, 0.000000, 0.000000, 0.999999, 0.999999, 0.999999, 0.726116, 0.752810, 0.383964, 0.825011, 0.700793, 0.760239, 0.475060],
                'X_3': [0.532570, 0.654825, 0.000000, 0.401482, 0.853396, 0.504106, 0.065550, 0.043440, 0.487088, 0.010544, 0.066701, 0.040946, 0.496556],
                'y': [-0.02558125028420759, -0.18278605326543287, -0.17457608290074642, -0.05784183677328758, -0.06927592824625894, -0.061735329132434506,
                      -0.032249639801471316, -0.0401865725216515, -0.016598983479561742, -0.10468817404316187, -0.04526438317909011, -0.060318148767364806,
                      -0.014783982680863506]
}

new_points_4 = {'X_1': [0.405882, 0.000000, 0.000000, 0.373764, 0.433724, 0.406854, 0.388248, 0.371119, 0.393943, 0.452111, 0.366221, 0.389802, 0.396291],
                'X_2': [0.444943, 0.354662, 0.999999, 0.431364, 0.476452, 0.385962, 0.431328, 0.410179, 0.415506, 0.434320, 0.423523, 0.415742, 0.392344],
                'X_3': [0.359978, 0.999999, 0.999999, 0.476888, 0.403076, 0.381346, 0.370494, 0.299391, 0.374950, 0.156132, 0.384173, 0.378980, 0.377939],
                'X_4': [0.453161, 0.999999, 0.000000, 0.436882, 0.425577, 0.452259, 0.438133, 0.431580, 0.432861, 0.432663, 0.401100, 0.403697, 0.399991],
                'y': [-0.21239375498335233, -35.25350633281045, -40.34837615277241, -0.7759995242242783, -0.7072513351996004, -0.4150781334340077,
                      0.3004715374918807, -0.8038788415865876, 0.4322660657877404, -4.084728550879301, 0.4911800814618634, 0.35085838324996077,
                      0.05919371281027397]
}

new_points_5 = {'X_1': [0.383238, 0.975604, 0.013159, 0.999999, 0.999999, 0.999999, 0.999993, 0.999989, 0.999929, 0.999999, 0.000000, 0.001290, 0.000000],
                'X_2': [0.834729, 0.000000, 0.000000, 0.996144, 0.999999, 0.999999, 0.999992, 0.999998, 0.999995, 0.999999, 0.592098, 0.063996, 0.000000],
                'X_3': [0.934027, 0.999999, 0.492408, 0.999999, 0.999999, 0.999999, 0.999996, 0.999997, 0.999996, 0.999999, 0.999999, 0.000000, 0.000000],
                'X_4': [0.989612, 0.999999, 0.999999, 0.999999, 0.000000, 0.999999, 0.999999, 0.999610, 0.999996, 0.667984, 0.000000, 0.000000, 0.000000],
                'y': [2166.44812278176, 4120.008867028152, 238.34246646122324, 8588.299663342219, 4440.480873479282, 8662.405001248297, 8662.095012865859,
                      8654.622491525643, 8660.855242684243, 5194.030278905406, 294.66757534036725, 163.03935420224883, 163.1225]
}

new_points_6 = {'X_1': [0.276498, 0.013491, 0.013493, 0.002056, 0.169639, 0.992514, 0.415368, 0.409265, 0.415299, 0.412965, 0.417732, 0.416140, 0.404655],
                'X_2': [0.326839, 0.142080, 0.142080, 0.224409, 0.360740, 0.273169, 0.368431, 0.359081, 0.371680, 0.406694, 0.350434, 0.365693, 0.311920],
                'X_3': [0.436366, 0.000000, 0.000000, 0.488931, 0.519767, 0.505759, 0.581641, 0.634220, 0.646578, 0.827087, 0.641293, 0.655422, 0.641710],
                'X_4': [0.728854, 0.959817, 0.000000, 0.999999, 0.636135, 0.650832, 0.720293, 0.714729, 0.709517, 0.875387, 0.721711, 0.688427, 0.780653],
                'X_5': [0.010656, 0.999999, 0.000000, 0.000000, 0.133508, 0.000000, 0.005775, 0.000002, 0.000005, 0.000000, 0.033030, 0.000000, 0.001035],
                'y': [-0.4985197741768271, -2.4146845386359046, -2.1345852906099454, -1.0623213289871172, -0.5835396869747151, -1.1083480245361563,
                      -0.23019435016931042, -0.12969609697285114, -0.24340470931636277, -0.44510002428166273, -0.13847855744672427, -0.1980776521439728,
                      -0.13294479403399193]
}

new_points_7 = {'X_1': [0.036258, 0.436487, 0.000000, 0.000000, 0.000000, 0.000000, 0.030578, 0.030735, 0.031348, 0.000000, 0.034243, 0.089868, 0.000000],
                'X_2': [0.355375, 0.693777, 0.961068, 0.255218, 0.163742, 0.323641, 0.321820, 0.330349, 0.344293, 0.276358, 0.325832, 0.174443, 0.144108],
                'X_3': [0.364856, 0.896067, 0.648265, 0.000180, 0.273869, 0.285495, 0.350456, 0.260900, 0.325538, 0.999999, 0.333470, 0.587825, 0.482899],
                'X_4': [0.170074, 0.000000, 0.345219, 0.000000, 0.060047, 0.003733, 0.174993, 0.136585, 0.161355, 0.243337, 0.158904, 0.221803, 0.291312],
                'X_5': [0.329410, 0.335648, 0.323192, 0.319644, 0.320778, 0.112884, 0.364052, 0.339946, 0.343333, 0.345796, 0.350354, 0.294941, 0.283232],
                'X_6': [0.718265, 0.999999, 0.850726, 0.888761, 0.723038, 0.723291, 0.715547, 0.738712, 0.723908, 0.732570, 0.719063, 0.914943, 0.736964],
                'y': [2.1575945951499675, 0.423476961132087, 0.2006962219221917, 0.5751749221572217, 1.5668871369110176, 0.6064780751588998, 2.1570308153425977,
                      1.8245854999439983, 2.0404088731162155, 1.2295012887269074, 2.0925081082148163, 1.8647417017183727, 2.6676534121873305]
}

new_points_8 = {'X_1': [0.169887, 0.000000, 0.999999, 0.125940, 0.216115, 0.000000, 0.012659, 0.072038, 0.059106, 0.134824, 0.080347, 0.101860, 0.090612],
                'X_2': [0.153636, 0.999999, 0.000000, 0.000000, 0.228658, 0.105528, 0.147093, 0.157484, 0.166751, 0.177481, 0.157246, 0.178682, 0.177043],
                'X_3': [0.000000, 0.000000, 0.000000, 0.389243, 0.157862, 0.000000, 0.129782, 0.057804, 0.103746, 0.142423, 0.108025, 0.112263, 0.119144],
                'X_4': [0.000000, 0.999999, 0.000000, 0.000000, 0.201830, 0.000000, 0.148919, 0.238465, 0.184476, 0.151052, 0.163591, 0.170320, 0.153972],
                'X_5': [0.948348, 0.999999, 0.999999, 0.999999, 0.999999, 0.416512, 0.798106, 0.779863, 0.832435, 0.808657, 0.901498, 0.772439, 0.796371],
                'X_6': [0.136260, 0.164058, 0.876566, 0.023204, 0.992794, 0.010808, 0.526084, 0.558431, 0.497207, 0.475211, 0.496718, 0.492412, 0.497251],
                'X_7': [0.000000, 0.000000, 0.000000, 0.000000, 0.297374, 0.077007, 0.198873, 0.130388, 0.201309, 0.227817, 0.221293, 0.206638, 0.210116],
                'X_8': [0.346808, 0.878849, 0.005403, 0.004949, 0.458802, 0.136052, 0.633915, 0.351691, 0.981694, 0.396095, 0.506870, 0.613670, 0.546792],
                'y': [9.6872974229276, 8.2636708961534, 8.007147288400601, 9.3892906387764, 9.6780295048451, 9.5402025745116, 9.9839336060015,
                      9.9554430712014, 9.9780120800159, 9.98999801097, 9.990605800596, 9.9972695226865, 9.9982211542111]
}

In [7]:
# Turning dictionaries of weekly results into dataframes 

new_points_1_df = pd.DataFrame(new_points_1)
new_points_2_df = pd.DataFrame(new_points_2)
new_points_3_df = pd.DataFrame(new_points_3)
new_points_4_df = pd.DataFrame(new_points_4)
new_points_5_df = pd.DataFrame(new_points_5)
new_points_6_df = pd.DataFrame(new_points_6)
new_points_7_df = pd.DataFrame(new_points_7)
new_points_8_df = pd.DataFrame(new_points_8)

# Appending weekly results to initial data

df_function_1 = pd.concat([dfs['function_1'], new_points_1_df], ignore_index=True)
df_function_2 = pd.concat([dfs['function_2'], new_points_2_df], ignore_index=True)
df_function_3 = pd.concat([dfs['function_3'], new_points_3_df], ignore_index=True)
df_function_4 = pd.concat([dfs['function_4'], new_points_4_df], ignore_index=True)
df_function_5 = pd.concat([dfs['function_5'], new_points_5_df], ignore_index=True)
df_function_6 = pd.concat([dfs['function_6'], new_points_6_df], ignore_index=True)
df_function_7 = pd.concat([dfs['function_7'], new_points_7_df], ignore_index=True)
df_function_8 = pd.concat([dfs['function_8'], new_points_8_df], ignore_index=True)